In [0]:

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import avg, count, round, col

In [0]:
# Azure Event Hubs Connection String, Event Hub Namespace, and name

CONNECTION_STR = dbutils.secrets.get(scope="azure-keyvault-databricks-scope", key="key-vault-secret-event-hub")


EVENTHUB_NAME = "evnt_hub_store"

EVENTHUB_NS = 'evnt-hub-stream-ns'

connectionConf = {
  'eventhubs.connectionString' : spark.conf.set('spark.eventhubs.connectionString', CONNECTION_STR),
  'eventhubs.name': EVENTHUB_NAME
}

In [0]:
# get the catalog assigned to your current workspace/session
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]

# Query the metadata
df = spark.sql(f"DESCRIBE CATALOG EXTENDED {current_catalog}")

# extract the underlying storage location path
storage_row = df.filter(df["info_name"] == "Storage Location").collect()

if storage_row:
    unity_catalog_storage = storage_row[0]["info_value"].split('_')[0].rstrip('/')

    print(f"{unity_catalog_storage}")
else:
    print("No specific catalog storage found.")


In [0]:

# Default storage location for catalog
storage_location = unity_catalog_storage

spark.sql("DROP CATALOG IF EXISTS weather_data_stream CASCADE")

# Create Unity Catalog
spark.sql(f"CREATE CATALOG weather_data_stream MANAGED LOCATION '{storage_location}'")

# Create bronze layer schema
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_data_stream.bronze")

# Create silver layer schema
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_data_stream.silver")

# Create gold layer schema
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_data_stream.gold")

# Create bronze layer volume for checkpoint
spark.sql("CREATE VOLUME IF NOT EXISTS weather_data_stream.bronze.bronze_volume")

# Create silver layer volume for checkpoint
spark.sql("CREATE VOLUME IF NOT EXISTS weather_data_stream.silver.silver_volume")

# Create gold layer volume for checkpoint
spark.sql("CREATE VOLUME IF  NOT EXISTS weather_data_stream.gold.gold_volume")

In [0]:
# # 1. Fetch your active notebook user identity automatically
# current_user_email = spark.sql("SELECT current_user()").collect()[0][0]
# # print(f"Assigning permissions to: {current_user_email}")

# # 2. Grant permissions on the Catalog
# spark.sql(f"GRANT USE CATALOG ON CATALOG weather_data_stream TO `{current_user_email}`")

# # 3. Grant permissions on the Schema
# spark.sql(f"GRANT USE SCHEMA, CREATE TABLE ON SCHEMA weather_data_stream.bronze TO `{current_user_email}`")

# # 4. Grant permissions on the Volume where your checkpoints are saved
# spark.sql(f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME weather_data_stream.bronze.bronze_volume TO `{current_user_email}`")

# print("All Unity Catalog privileges granted successfully!")


In [0]:
df_bronze = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{EVENTHUB_NS}.servicebus.windows.net:9093")
    .option("subscribe", f"{EVENTHUB_NAME}")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.ssl.endpoint.identification.algorithm", "https")
    .option("kafka.sasl.jaas.config", f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{CONNECTION_STR}";')
    .load())


In [0]:
import time

run_id = int(time.time())
checkpoint_path = f"/Volumes/weather_data_stream/bronze/bronze_volume/stream/path_{run_id}"
table_name = "weather_data_stream.bronze.bronze_data"



# Start the stream to the table
query = df_bronze.writeStream \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .format("delta") \
    .toTable(table_name)




In [0]:
%sql
SELECT * FROM weather_data_stream.bronze.bronze_data

In [0]:
# Define the JSON schema
json_schema = StructType([
    StructField("city", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("pressure", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("cloud_cover", DoubleType(), True),
    StructField("weather_condition", StringType(), True),
    StructField("timestamp", StringType(), True)
])

In [0]:
# Reading streaming data from the Delta table
silver_df = spark.readStream\
    .format("delta")\
    .table("weather_data_stream.bronze.bronze_data")\
    .withColumn("body", col("value").cast("string"))\
    .withColumn("body", from_json(col("body"), json_schema))



# Casting all columns to string
df_string = silver_df.select(
    col("body.city").cast("string").alias("city"),
    col("body.latitude").cast("string").alias("latitude"),
    col("body.longitude").cast("string").alias("longitude"),
    col("body.temperature").cast("string").alias("temperature"),
    col("body.humidity").cast("string").alias("humidity"),
    col("body.wind_speed").cast("string").alias("wind_speed"),
    col("body.pressure").cast("string").alias("pressure"),
    col("body.precipitation").cast("string").alias("precipitation"),
    col("body.cloud_cover").cast("string").alias("cloud_cover"),
    col("body.weather_condition").cast("string").alias("weather_condition"),
    col("body.timestamp").cast("string").alias("timestamp")
)

# display(df_string)
run_id = int(time.time())
checkpoint_path = f"/Volumes/weather_data_stream/silver/silver_volume/stream/path_{run_id}"

# Writing the streaming data to the Delta table in append mode
df_string.writeStream\
    .option("checkpointLocation", checkpoint_path ) \
    .outputMode("append")\
    .format("delta")\
    .toTable("weather_data_stream.silver.silver_data")



In [0]:
%sql
SELECT * FROM weather_data_stream.silver.silver_data;

In [0]:
# Reading streaming data from the Silver table
df_gold = spark.readStream \
    .format("delta") \
    .table("weather_data_stream.silver.silver_data")

# Performing advanced aggregation and rounding to 2 decimal places
df_aggregated = df_gold.groupBy(col("city")).agg(
    round(avg(col("temperature")), 2).alias("avg_temperature"),
    round(avg(col("humidity")), 2).alias("avg_humidity"),
    round(avg(col("wind_speed")), 2).alias("avg_wind_speed"),
    count("*").alias("record_count")
)

# display(df_aggregated)

# Writing the aggregated streaming data to the Delta table in append mode
checkpoint_path = f'/Volumes/weather_data_stream/gold/gold_volume/stream/stream/path_{run_id}'
df_aggregated.writeStream \
    .option("checkpointLocation", checkpoint_path) \
    .outputMode("complete") \
    .format("delta") \
    .toTable("weather_data_stream.gold.gold_data")



 

In [0]:
%sql

select * from weather_data_stream.gold.gold_data